# Lab 2: Trotterization of the Hubbard model

**Goal.** In this lab you will build, step by step and *by yourself*, a function that turns a Hamiltonian written as a sum of Pauli strings into a quantum circuit approximating the time evolution $e^{-iHt}$. At the end, you will check your work against the Trotterization tools of `myqlm-fermion`.

**Our test system.** The Hubbard model as discussed during lecture, on a $2\times 2$ lattice (8 spin-orbitals, hence 8 qubits):

$$H = \sum_{i,j,\sigma} t_{ij}\, c_{i\sigma}^\dagger c_{j\sigma} + U \sum_i n_{i\uparrow} n_{i\downarrow}$$

**Plan**

| Part | Content |
|---|---|
| 0 | Setup and conventions |
| 1 | From fermions to Pauli strings (Jordan-Wigner, by hand) |
| 2 | The exact answer (classical reference) |
| 3 | The building block: the circuit for $e^{-i\theta P}$ |
| 4 | First-order Trotter |
| 5 | Error analysis |
| 6 | Second-order Trotter |
| 7 | Verification against `myqlm-fermion` |
| 8 | Physics: watching the Hubbard model evolve |


## Part 0: Setup and conventions

In [ ]:
!pip install myqlm myqlm-fermion

In [ ]:
import itertools
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from qat.lang.AQASM import Program, QRoutine, H, X, RX, RZ, CNOT
from qat.qpus import PyLinalg
from qat.fermion.hamiltonians import make_hubbard_model
from qat.fermion.transforms import transform_to_jw_basis
from qat.fermion.trotterisation import make_trotterisation_routine

plt.rcParams["figure.figsize"] = (6, 4)

### Conventions used throughout this lab

* **Qubit ordering.** Qubit 0 is the *leftmost* (most significant) bit: the basis state $|q_0 q_1 \dots q_{n-1}\rangle$ has index $\sum_k q_k 2^{n-1-k}$. Matrices are built with `np.kron` in the order qubit 0, qubit 1, ...
* **Occupation.** $|0\rangle$ = empty orbital, $|1\rangle$ = occupied orbital.
* **Spin-orbital labelling.** Orbital index $p = 2k + \sigma$, with $k$ the site and $\sigma\in\{0 (\uparrow), 1 (\downarrow)\}$. Orbital/qubit $p$ therefore sits on site $p // 2$.
* **Rotation gate.** `RZ(theta)` $= e^{-i\theta Z/2}$.
* **Lattice.** Sites are numbered as follows, with hopping $t_{ij} = -1$ on each bond:

```
   site 0 --- site 1          qubits (0,1) --- (2,3)
     |           |               |               |
   site 2 --- site 3          qubits (4,5) --- (6,7)
```

Helper functions (those are given here to help you: read them, but do not modify them).

In [ ]:
# ---- Pauli matrices and dense matrices of Pauli strings -----------------------------------
I2 = np.eye(2, dtype=complex)
PAULI = {
    "I": I2,
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}


def pauli_matrix(pauli_string):
    '''Dense matrix of a Pauli string such as "XZIY" (qubit 0 is the leftmost factor).'''
    mat = np.array([[1]], dtype=complex)
    for letter in pauli_string:
        mat = np.kron(mat, PAULI[letter])
    return mat


def pauli_sum_matrix(terms):
    '''Dense matrix of sum_k c_k P_k, where terms = [(c_k, "XZI..."), ...].'''
    return sum(c * pauli_matrix(s) for c, s in terms)


# ---- Running circuits and extracting what they do ------------------------------------------
def append_routine(parent, child):
    '''Apply `child` (a QRoutine) inside `parent` on wires 0..child.arity-1.

    Note: the arity of a QRoutine is (highest wire used) + 1, so a routine acting on e.g. qubits
    {0, 2} has arity 3. Skip empty routines.'''
    if child is not None and child.arity > 0:
        parent.apply(child, list(range(child.arity)))


def run_state(routine, n_qubits, init_bits):
    '''Prepare the basis state `init_bits` (list of 0/1), apply `routine`, return the state vector.'''
    prog = Program()
    qbits = prog.qalloc(n_qubits)
    for i, bit in enumerate(init_bits):
        if bit:
            prog.apply(X, qbits[i])
    if routine is not None and routine.arity > 0:
        prog.apply(routine, qbits[: routine.arity])
    result = PyLinalg().submit(prog.to_circ().to_job())
    psi = np.zeros(2**n_qubits, dtype=complex)
    for sample in result.raw_data:
        psi[sample.state.int] = sample.amplitude
    return psi


def circuit_unitary(routine, n_qubits):
    '''Full unitary matrix of a routine (obtained by running it on every basis state).'''
    U = np.zeros((2**n_qubits, 2**n_qubits), dtype=complex)
    for k in range(2**n_qubits):
        bits = [(k >> (n_qubits - 1 - i)) & 1 for i in range(n_qubits)]
        U[:, k] = run_state(routine, n_qubits, bits)
    return U


def gate_count(routine, n_qubits):
    '''Total number of elementary gates in the (inlined) circuit.'''
    prog = Program()
    qbits = prog.qalloc(n_qubits)
    prog.apply(routine, qbits[: routine.arity])
    return len(prog.to_circ().ops)

## Part 1: From fermions to Pauli strings

### The Hamiltonian, as `myqlm-fermion` sees it

We first ask the library for the Hubbard Hamiltonian and its Jordan-Wigner image. This is our **reference**: in the next exercises you will rebuild the same thing yourself.

In [ ]:
n_sites, n_qubits = 4, 8

# Hopping matrix for the 2x2 lattice (basically says which sites are connected to which)
t_mat = np.zeros((n_sites, n_sites))
for i, j in [(0, 1), (2, 3), (0, 2), (1, 3)]:
    t_mat[i, j] = t_mat[j, i] = -1.0
U_int = 4.0

H_fermion = make_hubbard_model(t_mat, U_int, 0.0)   # second-quantized form
H_lib = transform_to_jw_basis(H_fermion)             # Jordan-Wigner: sum of Pauli strings


def spin_hamiltonian_to_pauli_list(spin_ham, n):
    '''Convert a myQLM SpinHamiltonian into ([(coeff, "XZIX..."), ...], constant).'''
    terms = []
    for term in spin_ham.terms:
        string = ["I"] * n
        for qubit, letter in zip(term.qbits, term.op):
            string[qubit] = letter
        terms.append((float(np.real(term.coeff)), "".join(string)))
    return terms, float(np.real(spin_ham.constant_coeff))


terms, constant = spin_hamiltonian_to_pauli_list(H_lib, n_qubits)
print(f"{len(terms)} Pauli terms, plus a constant {constant} (a pure global phase in e^(-iHt)).\n")
for c, s in terms:  
    print(f"{c:+.2f}  {s}")


Look at the strings: some are long chains of $Z$'s sandwiched between two $X$'s or two $Y$'s, others are only $Z$'s.

### Exercise 1.1: the Jordan-Wigner annihilation operator

With our conventions ($|0\rangle$ empty), the annihilation operator of orbital $p$ among $n$ is

$$c_p = Z_0 \otimes \dots \otimes Z_{p-1} \otimes \sigma^-_p \otimes I_{p+1}\otimes\dots\otimes I_{n-1}, \qquad \sigma^- = |0\rangle\langle 1| = \tfrac12 (X + iY).$$

Write `annihilation_operator(p, n)`, returning the dense $2^n\times 2^n$ matrix of $c_p$.

In [ ]:
def annihilation_operator(p, n):
    '''Dense matrix of the Jordan-Wigner annihilation operator c_p on n orbitals.'''
    # YOUR CODE HERE (hint: build it factor by factor with np.kron, as in pauli_matrix)
    raise NotImplementedError

In [ ]:
# Test: canonical anticommutation relations {c_p, c_q^dagger} = delta_pq and {c_p, c_q} = 0
n_test = 4
c = [annihilation_operator(p, n_test) for p in range(n_test)]
Id = np.eye(2**n_test)
for p, q in itertools.product(range(n_test), repeat=2):
    anti = c[p] @ c[q].conj().T + c[q].conj().T @ c[p]
    assert np.allclose(anti, (p == q) * Id), f"{{c_{p}, c_{q}^dag}} is wrong"
    assert np.allclose(c[p] @ c[q] + c[q] @ c[p], 0), f"{{c_{p}, c_{q}}} is wrong"
print("Anticommutation relations satisfied.")

### Exercise 1.2: build the Hubbard matrix directly from fermionic operators

Using `annihilation_operator`, write `hubbard_matrix(t_mat, U)` that returns the dense matrix of

$$H = \sum_{i,j,\sigma} t_{ij}\, c_{i\sigma}^\dagger c_{j\sigma} + U \sum_i n_{i\uparrow} n_{i\downarrow}, \qquad n_{p} = c_p^\dagger c_p,$$

with the labelling $p = 2i+\sigma$. Your result should match the library's matrix `H_lib.get_matrix()`.

In [ ]:
def hubbard_matrix(t_mat, U):
    '''Dense matrix of the Hubbard Hamiltonian built directly from fermionic operators.'''
    n_sites = t_mat.shape[0]
    n = 2 * n_sites
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert np.allclose(hubbard_matrix(t_mat, U_int), H_lib.get_matrix()), "Your matrix differs from the library's."
print("Your Hubbard matrix matches myqlm-fermion's Jordan-Wigner Hamiltonian.")

### Exercise 1.3: the Pauli form of a hopping term

Take $p<q$. Show on paper that

$$c_p^\dagger c_q + c_q^\dagger c_p = \tfrac12\left( X_p Z_{p+1}\cdots Z_{q-1} X_q + Y_p Z_{p+1}\cdots Z_{q-1} Y_q\right).$$

Then write `hopping_pauli_terms(p, q, n)` returning the list `[(coeff, pauli_string), ...]` (strings of full length `n`) of this operator, and check it against the matrices.

In [ ]:
def hopping_pauli_terms(p, q, n):
    '''Pauli decomposition of c_p^dag c_q + c_q^dag c_p (p < q) on n qubits.'''
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
for p, q in [(0, 1), (0, 2), (1, 4), (2, 7)]:
    cp, cq = annihilation_operator(p, 8), annihilation_operator(q, 8)
    exact = cp.conj().T @ cq + cq.conj().T @ cp
    assert np.allclose(pauli_sum_matrix(hopping_pauli_terms(p, q, 8)), exact), f"Wrong for (p, q) = ({p}, {q})"
print("Hopping terms are correct.")

**Questions (answer in a markdown cell)**

1. In the lattice above, sites 0 and 2 are neighbours. Which qubits are involved in the corresponding hopping terms, and why do they include "Z" operators in between?
2. Which Pauli strings in the list `terms` (the full Hubbard Hamiltonian computed from `H_lib` above) come from the interaction $U n_{\uparrow}n_{\downarrow}$? What do they have in common?

## Part 2: The exact answer

Before simulating anything on a "quantum computer", we want a classical reference to compare with. With $n=8$ qubits, the matrix is only $256\times256$: we can exponentiate it exactly.

Throughout the rest of the lab, we drop the constant term of $H$ (it only produces a global phase) and work with

$$H = \sum_k c_k P_k, \qquad P_k \text{ Pauli strings},$$

stored in the list `terms` above.

**Initial state.** A Néel-like checkerboard: spin-up electrons on sites 0 and 3, spin-down electrons on sites 1 and 2.

In [ ]:
H_exact = pauli_sum_matrix(terms)            # without the constant

init_bits = [1, 0,   0, 1,   0, 1,   1, 0]   # (site0: up) (site1: down) (site2: down) (site3: up)
psi0 = run_state(None, n_qubits, init_bits)  # state vector of this basis state


def number_operator(p, n=n_qubits):
    cp = annihilation_operator(p, n)
    return cp.conj().T @ cp


n_up_site0 = number_operator(0)                          # occupation of orbital 0 (site 0, spin up)
double_occ_site0 = number_operator(0) @ number_operator(1)   # double occupancy of site 0
print("Initial <n_0up> =", np.real(psi0.conj() @ n_up_site0 @ psi0))

### Exercise 2.1: exact dynamics

Compute, for `times = np.linspace(0, 3, 16)`, the exact expectation values $\langle n_{0\uparrow}\rangle(t)$ and $\langle D_0\rangle(t)=\langle n_{0\uparrow}n_{0\downarrow}\rangle(t)$ in the state $e^{-iHt}|\psi_0\rangle$, using `scipy.linalg.expm`. Store them in `nup_exact` and `D_exact`, then plot them.

In [ ]:
times = np.linspace(0, 3, 16)
nup_exact = np.zeros_like(times)
D_exact = np.zeros_like(times)

# YOUR CODE HERE
raise NotImplementedError

plt.plot(times, nup_exact, "o-", label=r"$\langle n_{0\uparrow}\rangle$")
plt.plot(times, D_exact, "s-", label=r"$\langle D_0\rangle$")
plt.xlabel("t"); plt.legend(); plt.show()

In [ ]:
assert abs(nup_exact[0] - 1) < 1e-12 and abs(D_exact[0]) < 1e-12, "Check the value at t = 0."
assert abs(nup_exact[-1] - 1) > 1e-2, "Nothing is moving: check that you apply exp(-iHt) for each t."
print("✅ Exact reference computed.")

## Interlude: what is Trotterization, and where does it come from?

**The problem.** We want the circuit for the time evolution $e^{-iHt}$. For our Hamiltonian, $H=\sum_{k=1}^{K} c_kP_k$ is a sum of $K$ Pauli strings. Exponentiating a *single* Pauli string is easy (that is Part 3 below). Exponentiating the *whole sum* is not: we would need to diagonalize $H$, which is exactly the exponentially hard classical problem we hoped the quantum computer would avoid.

**Why we cannot simply split the exponential.** For numbers, $e^{a+b}=e^ae^b$. For operators this only holds if they commute. In general, the Baker-Campbell-Hausdorff formula gives

$$e^{A}e^{B}=\exp\!\Big(A+B+\tfrac12[A,B]+\tfrac1{12}\big([A,[A,B]]+[B,[B,A]]\big)+\dots\Big),\qquad [A,B]=AB-BA.$$

So $e^{A}e^{B}\neq e^{A+B}$ as soon as $[A,B]\neq0$, and the discrepancy is controlled by the commutators.

**The key observation (Trotter, 1959).** The commutator terms are of higher order in the size of the operators. Take $A=-iX\,t/n$ and $B=-iY\,t/n$, with $n$ large so that these operators are small. Then

$$e^{-i(X+Y)t/n}=e^{-iXt/n}\,e^{-iYt/n}+O\!\big(t^2/n^2\big).$$

The error of one *short* step is of order $(t/n)^2$. Repeating the step $n$ times, we get

$$e^{-i(X+Y)t}=\Big(e^{-iXt/n}\,e^{-iYt/n}\Big)^{n}+O\!\Big(\frac{t^2}{n}\Big),$$

because $n$ steps accumulate $n\times(t/n)^2=t^2/n$. More precisely, the error is bounded by $\frac{t^2}{2n}\,\|[X,Y]\|$. **This is the Lie-Trotter formula**, also called first-order Trotterization: the error goes to zero as the number of steps $n\to\infty$, and it vanishes identically when the terms commute.

For $K$ terms, a step is just $\prod_{k}e^{-ic_kP_k\,t/n}$, and the error involves all the commutators $[c_jP_j,\,c_kP_k]$. Two Pauli strings either commute or anticommute, so this can be checked term by term.

**Higher orders (Suzuki, 1976-1990).** The leading error comes from the $\tfrac12[A,B]$ term. If we apply the factors in a *symmetric* order, for example half a step of $A$, a full step of $B$, half a step of $A$ (**Strang splitting**), the leading commutator cancels and the error becomes $O(t^3/n^2)$. Suzuki showed how to build, *recursively*, formulas of any even order $2k$, at the price of more exponentials per step (the number grows like $5^{k-1}$). This family of product formulas is called **Trotter-Suzuki** expansions.

**Why this matters for quantum computing (Lloyd, 1996).** Each factor $e^{-ic_kP_k\,dt}$ is a simple, local-enough circuit. A product formula therefore turns "simulate $e^{-iHt}$" into "apply a sequence of simple gates". The trade-off is:

* more steps $n$ (or a higher order) $\Rightarrow$ smaller Trotter error,
* but also more gates $\Rightarrow$ deeper circuits, which matters a lot on noisy hardware.

Parts 4-6 below explore exactly this trade-off on the Hubbard model.

## Part 3: The building block, $e^{-i\theta P}$ for a Pauli string $P$

The Hamiltonian is $H=\sum_k c_kP_k$. Every Trotter formula is a product of exponentials of *single* Pauli strings, so this is the one gadget we need.

**Recipe** for a Pauli string $P$ acting non-trivially on qubits $q_1<q_2<\dots<q_m$:

1. **Basis change** so that every non-trivial factor becomes a $Z$: apply $H$ on qubits with an $X$, and `RX(pi/2)` on qubits with a $Y$.
2. **Parity computation**: a ladder of CNOTs $q_1\to q_2$, $q_2\to q_3$, ..., $q_{m-1}\to q_m$ puts the parity of the $m$ bits on qubit $q_m$.
3. **Rotation**: `RZ(2*theta)` on qubit $q_m$ (remember `RZ(a)` $= e^{-iaZ/2}$).
4. **Undo** steps 2 and 1 in reverse order (note `RX(-pi/2)` undoes `RX(pi/2)`).

Qubits where $P$ has an identity are simply skipped.

> *Why does this work?* After steps 1-2, $P$ has been mapped to a single $Z$ on qubit $q_m$ by a unitary $V$, i.e. $P = V^\dagger Z_{q_m} V$, and therefore $e^{-i\theta P} = V^\dagger e^{-i\theta Z_{q_m}} V$.

### Exercise 3.1: write `pauli_exponential(pauli_string, theta)`

It must return a `QRoutine` implementing $e^{-i\theta P}$ (or `None` if the string is the identity). Use `append_routine` if you need to nest routines.

In [ ]:
def pauli_exponential(pauli_string, theta):
    '''QRoutine implementing exp(-i * theta * P), where P is the Pauli string `pauli_string`.'''
    routine = QRoutine()
    active = [q for q, letter in enumerate(pauli_string) if letter != "I"]
    if not active:
        return None
    # YOUR CODE HERE
    raise NotImplementedError
    return routine

In [ ]:
# Test against the exact matrix exponential, including strings with identities in the middle
rng = np.random.default_rng(1234)
for s in ["Z", "X", "Y", "ZZ", "XZI", "IYZ", "XYX", "YIY", "ZIZ"]:
    theta = rng.uniform(-1.5, 1.5)
    n = len(s)
    U_circuit = circuit_unitary(pauli_exponential(s, theta), n)
    U_exact = expm(-1j * theta * pauli_matrix(s))
    assert np.allclose(U_circuit, U_exact), f"Wrong unitary for P = {s} (theta = {theta:.3f})"
print("pauli_exponential is correct (global phase included).")

**Question.** For a Pauli string of weight $w$ (i.e. $w$ non-identity factors), how many CNOT gates does your circuit use? What does this imply for the long $Z$-strings generated by Jordan-Wigner?

## Part 4: First-order Trotter

We want $e^{-iHt}$ with $H=\sum_k c_kP_k$, but the $P_k$ do not commute, so $e^{-iHt}\neq\prod_k e^{-ic_kP_kt}$. The **Lie-Trotter formula** nevertheless gives, for $n$ steps,

$$e^{-iHt} \approx \left(\prod_{k} e^{-i c_k P_k\, t/n}\right)^{n}.$$

### Exercise 4.1: write `trotter_first_order(terms, t, n_steps)`

It returns a `QRoutine` with the $n$ steps chained, each step applying `pauli_exponential(P_k, c_k * dt)` for all $k$ in the order of `terms` ($dt=t/n$). Use `append_routine(routine, sub)` to nest the gadgets.

In [ ]:
def trotter_first_order(terms, t, n_steps):
    '''First-order Trotter circuit for exp(-i t H), H = sum_k c_k P_k, terms = [(c_k, P_k), ...].'''
    routine = QRoutine()
    dt = t / n_steps
    # YOUR CODE HERE
    raise NotImplementedError
    return routine

In [ ]:
# Test 1: commuting terms => Trotterization is exact, even with a single step
commuting = [(0.7, "ZZI"), (0.3, "IZZ"), (-0.4, "ZII")]
U_circ = circuit_unitary(trotter_first_order(commuting, 1.3, 1), 3)
assert np.allclose(U_circ, expm(-1j * 1.3 * pauli_sum_matrix(commuting))), "Commuting terms should give an exact result."

# Test 2: non-commuting terms => error for 1 step, and it shrinks when the number of steps grows
noncomm = [(1.0, "X"), (1.0, "Z")]
U_exact = expm(-1j * 1.0 * pauli_sum_matrix(noncomm))
err = lambda ns: np.linalg.norm(circuit_unitary(trotter_first_order(noncomm, 1.0, ns), 1) - U_exact)
e1, e500 = err(1), err(500)
assert e1 > 0.1 and e500 < 0.01, f"Unexpected errors: {e1:.3f} (1 step), {e500:.4f} (500 steps)"
print(f"Trotter works. Single-qubit toy model: error {e1:.3f} with 1 step, {e500:.4f} with 500 steps.")

## Part 5: Error analysis

Time to look at the Hubbard model. Our error measure is the distance between the Trotterized state and the exact one at time $t=1$:

$$\varepsilon = \left\| e^{-iHt}|\psi_0\rangle - U_{\text{Trotter}}|\psi_0\rangle \right\|.$$

(Both use the same Hamiltonian without constant, so there is no spurious global phase.)

Below, `trotter_circuit` is a small wrapper that selects the order (the second-order case will be filled in during Part 6).

In [ ]:
def trotter_circuit(terms, t, n_steps, order=1):
    if order == 1:
        return trotter_first_order(terms, t, n_steps)
    if order == 2:
        return trotter_second_order(terms, t, n_steps)
    raise ValueError("order must be 1 or 2")


def trotter_error(terms_used, t, n_steps, order=1):
    '''State error at time t when Trotterizing the Hamiltonian given by `terms_used`.'''
    exact_state = expm(-1j * pauli_sum_matrix(terms_used) * t) @ psi0
    trotter_state = run_state(trotter_circuit(terms_used, t, n_steps, order), n_qubits, init_bits)
    return np.linalg.norm(trotter_state - exact_state)


t_final = 1.0
steps_list = [1, 2, 4, 8, 16, 32]
errors_1 = [trotter_error(terms, t_final, ns, order=1) for ns in steps_list]

plt.loglog(steps_list, errors_1, "o-", label="first order")
plt.xlabel("number of Trotter steps n"); plt.ylabel("state error"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

slope = np.polyfit(np.log(steps_list[2:]), np.log(errors_1[2:]), 1)[0]
print(f"Fitted slope (last points): {slope:.2f}")

**Questions**

1. What slope do you expect from the theory for a first-order formula, i.e. how does the error scale with $n$ at fixed $t$? Does your plot agree?
2. At $n=1$ the error is close to its maximum possible value. Why is a single step useless here, with $t=1$ and couplings of order 1?

### Which terms are responsible for the error?

Let us split the Hamiltonian into its diagonal part (only $I$ and $Z$) and its hopping part.

In [ ]:
is_diagonal = lambda s: set(s) <= {"I", "Z"}
diag_terms = [(c, s) for c, s in terms if is_diagonal(s)]
hop_terms = [(c, s) for c, s in terms if not is_diagonal(s)]
print(f"{len(diag_terms)} diagonal terms, {len(hop_terms)} hopping terms")

for name, subset in [("diagonal (U) terms only", diag_terms), ("hopping terms only", hop_terms), ("full Hamiltonian", terms)]:
    print(f"{name:26s}", ["%.1e" % trotter_error(subset, 1.0, ns) for ns in (1, 4, 16)])

3. Why is the error of the diagonal part zero (to machine precision), whatever the number of steps?
4. Within the hopping terms, which pairs of terms commute and which do not? (Think about the two terms coming from the same bond, and about two bonds sharing a site.)

### Does the order of the terms matter?

In [ ]:
rng = np.random.default_rng(7)
ns = 4
print(f"default order        : {trotter_error(terms, 1.0, ns):.3f}")
print(f"diagonal terms first : {trotter_error(diag_terms + hop_terms, 1.0, ns):.3f}")
print(f"diagonal terms last  : {trotter_error(hop_terms + diag_terms, 1.0, ns):.3f}")
shuffled = [trotter_error([terms[k] for k in rng.permutation(len(terms))], 1.0, ns) for _ in range(5)]
print("5 random orders      :", [f"{e:.3f}" for e in shuffled])

5. The ordering changes the error but not the order of convergence. Why does the first-order error depend on the ordering (look at the commutator terms in the error expansion)?

## Part 6: Second-order (symmetric) Trotter

The **Strang / symmetric** formula applies half-steps forward, then the same half-steps in reverse order:

$$e^{-iHt}\approx\left(\prod_{k=1}^{K} e^{-ic_kP_k\,dt/2}\ \prod_{k=K}^{1} e^{-ic_kP_k\,dt/2}\right)^{n}, \qquad dt=t/n.$$

Its error scales as $O(t^3/n^2)$.

### Exercise 6.1: write `trotter_second_order(terms, t, n_steps)`

In [ ]:
def trotter_second_order(terms, t, n_steps):
    '''Second-order (symmetric) Trotter circuit for exp(-i t H).'''
    routine = QRoutine()
    dt = t / n_steps
    # YOUR CODE HERE
    raise NotImplementedError
    return routine

In [ ]:
errors_2 = [trotter_error(terms, t_final, ns, order=2) for ns in steps_list]
slope_1 = np.polyfit(np.log(steps_list[2:]), np.log(errors_1[2:]), 1)[0]
slope_2 = np.polyfit(np.log(steps_list[2:]), np.log(errors_2[2:]), 1)[0]
print(f"slopes: first order {slope_1:.2f}, second order {slope_2:.2f}")
assert -1.3 < slope_1 < -0.7, "First-order slope should be close to -1."
assert -2.4 < slope_2 < -1.7, "Second-order slope should be close to -2. Check that the second half-step is reversed."
print("Second order OK.")

plt.loglog(steps_list, errors_1, "o-", label=f"first order (slope {slope_1:.2f})")
plt.loglog(steps_list, errors_2, "s-", label=f"second order (slope {slope_2:.2f})")
plt.xlabel("number of Trotter steps n"); plt.ylabel("state error"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

### Accuracy vs. cost

On real hardware, what you pay for is the number of gates, not the number of steps. Let us plot the error as a function of the gate count.

In [ ]:
gates_1 = [gate_count(trotter_circuit(terms, t_final, ns, 1), n_qubits) for ns in steps_list]
gates_2 = [gate_count(trotter_circuit(terms, t_final, ns, 2), n_qubits) for ns in steps_list]

plt.loglog(gates_1, errors_1, "o-", label="first order")
plt.loglog(gates_2, errors_2, "s-", label="second order")
plt.xlabel("total gate count"); plt.ylabel("state error"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

**Questions**

1. For a target error of $10^{-1}$, which order is cheaper? And for $10^{-3}$? Extrapolate: what happens for a target error of $10^{-6}$?
2. A second-order step contains each term twice. In your circuit, the last term of the forward half-step and the first term of the backward half-step are the *same* exponential, applied back to back. How could you save gates? (Bonus: implement it.)

## Part 7: Verification against `myqlm-fermion`

`myqlm-fermion` provides `make_trotterisation_routine(hamiltonian, n_trotter_steps, final_time)`, a first-order Trotterization. It ignores the constant term of $H$ and applies the terms in the order stored in the Hamiltonian, which is also the order of your `terms` list. So the two circuits should produce the same state.

In [ ]:
n_steps = 4
library_circuit = make_trotterisation_routine(H_lib, n_steps, t_final)
psi_library = run_state(library_circuit, n_qubits, init_bits)
psi_mine = run_state(trotter_first_order(terms, t_final, n_steps), n_qubits, init_bits)
print("|| library - mine ||  =", np.linalg.norm(psi_library - psi_mine))

# The same function accepts the fermionic Hamiltonian directly (Jordan-Wigner is its default mapping):
psi_fermionic = run_state(make_trotterisation_routine(H_fermion, n_steps, t_final), n_qubits, init_bits)
print("|| fermionic input - mine || =", np.linalg.norm(psi_fermionic - psi_mine))

The distance should be at the level of $10^{-15}$. Your hand-made function does exactly what the library does.

**Questions**

1. The library does not offer a second-order option in this function. How could you obtain one with *only* the library's first-order routine? (Hint: a symmetric step is a first-order step of length $dt/2$ followed by the same first-order step with the terms in *reversed* order.)
2. The library also accepts `method="bravyi-kitaev"` or `"parity"` instead of Jordan-Wigner. What would change in the Pauli strings, and in the CNOT count? Why must the *initial state preparation* change as well? (This is a nice thing to explore after the lab.)

## Part 8: Watching the Hubbard model evolve

Finally, let us use our circuits to reproduce the physics of Part 2: starting from the Néel-like state, how do $\langle n_{0\uparrow}\rangle$ and $\langle D_0\rangle$ evolve? We keep the Trotter step *fixed* ($dt=0.2$), so reaching a time $t$ takes $t/dt$ steps.

In [ ]:
dt = 0.2
times_trotter = times[times <= 3.0 + 1e-9]
results = {}
for order in (1, 2):
    nup, docc = [], []
    for t in times_trotter:
        n_steps_t = int(round(t / dt))
        if n_steps_t == 0:
            psi = psi0
        else:
            psi = run_state(trotter_circuit(terms, t, n_steps_t, order), n_qubits, init_bits)
        nup.append(np.real(psi.conj() @ n_up_site0 @ psi))
        docc.append(np.real(psi.conj() @ double_occ_site0 @ psi))
    results[order] = (np.array(nup), np.array(docc))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, idx, title, exact in [(axes[0], 0, r"$\langle n_{0\uparrow}\rangle$", nup_exact), (axes[1], 1, r"$\langle D_0\rangle$", D_exact)]:
    ax.plot(times, exact, "k-", lw=2, label="exact")
    ax.plot(times_trotter, results[1][idx], "o--", label=f"Trotter order 1 (dt={dt})")
    ax.plot(times_trotter, results[2][idx], "s--", label=f"Trotter order 2 (dt={dt})")
    ax.set_xlabel("t"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

**Questions**

1. At fixed $dt$, how do the two Trotter curves deviate from the exact one as $t$ grows? What does this tell you about the cost of simulating *long* times on a quantum computer?
2. How would you expect the Trotter error to change if you increase $U$? If you move to a $3\times 3$ lattice? (Think about the number of terms and the commutators.)

---

## Wrap-up and extensions

You have built a complete Trotterization pipeline: **fermionic Hamiltonian → Jordan-Wigner → Pauli strings → circuit**, validated at each level against an exact or library reference.

**Optional extensions**

* Implement the gate-saving merge of consecutive identical exponentials in the second-order formula and measure the gain.
* Group the hopping terms into sets of mutually commuting terms (e.g. bonds that share no site) and see how the error and the *circuit depth* change.
* Implement the fourth-order Suzuki formula recursively.
* Add a finite number of shots (`n_shots`) or a noise model, and compare the Trotter error with the statistical error.
* Try the Bravyi-Kitaev mapping (and fix the initial state accordingly).